In [1]:
import os
import pandas as pd
import sqlite3
import shutil
import time

In [2]:
def load_cleaned_data():
    """Load all cleaned CSV files into pandas DataFrames."""
    crss_acc = pd.read_csv("CRSS/cleaned_data/crss_accident_cleaned.csv")
    crss_veh = pd.read_csv("CRSS/cleaned_data/crss_vehicle_cleaned.csv")
    crss_per = pd.read_csv("CRSS/cleaned_data/crss_person_cleaned.csv")

    fars_acc = pd.read_csv("FARS/cleaned_data/fars_accident_cleaned.csv")
    fars_veh = pd.read_csv("FARS/cleaned_data/fars_vehicle_cleaned.csv")
    fars_per = pd.read_csv("FARS/cleaned_data/fars_person_cleaned.csv")

    return crss_acc, crss_veh, crss_per, fars_acc, fars_veh, fars_per

def write_to_database(conn, crss_acc, crss_veh, crss_per, fars_acc, fars_veh, fars_per):
    """Write cleaned DataFrames to the SQLite database as separate tables."""
    crss_acc.to_sql("crss_accident", conn, if_exists="replace", index=False)
    crss_veh.to_sql("crss_vehicle", conn, if_exists="replace", index=False)
    crss_per.to_sql("crss_person", conn, if_exists="replace", index=False)

    fars_acc.to_sql("fars_accident", conn, if_exists="replace", index=False)
    fars_veh.to_sql("fars_vehicle", conn, if_exists="replace", index=False)
    fars_per.to_sql("fars_person", conn, if_exists="replace", index=False)

def check_duplicates(conn):
    """Check for duplicate rows in CRSS and FARS combined tables based on unique identifiers."""

    crss_query = """
    SELECT *
    FROM crss_combined
    WHERE (CASENUM, VEH_NO, PER_NO) IN (
        SELECT CASENUM, VEH_NO, PER_NO
        FROM crss_combined
        GROUP BY CASENUM, VEH_NO, PER_NO
        HAVING COUNT(*) > 1
    )
    """
    df_crss = pd.read_sql_query(crss_query, conn)
    print("Number of duplicate records in CRSS:", len(df_crss))

    fars_query = """
    SELECT *
    FROM fars_combined
    WHERE (ID, VEH_NO, PER_NO) IN (
        SELECT ID, VEH_NO, PER_NO
        FROM fars_combined
        GROUP BY ID, VEH_NO, PER_NO
        HAVING COUNT(*) > 1
    )
    """
    df_fars = pd.read_sql_query(fars_query, conn)
    print("Number of duplicate records in FARS:", len(df_fars))
    
def export_tables(conn, output_dir="output"):
    os.makedirs(output_dir, exist_ok=True)

    # Export CSVs
    pd.read_sql("SELECT * FROM fars_combined", conn).to_csv(os.path.join(output_dir, "fars_combined.csv"), index=False)
    pd.read_sql("SELECT * FROM crss_combined", conn).to_csv(os.path.join(output_dir, "crss_combined.csv"), index=False)

    # Close connection first
    conn.close()

    # Wait briefly to ensure file locks are released (Windows-specific safeguard)
    time.sleep(1)

    # Then move the file
    src = "crss_fars_all.db"
    dst = os.path.join(output_dir, "crss_fars_all.db")

    # If a previous file exists in output, remove it before moving
    if os.path.exists(dst):
        os.remove(dst)

    shutil.move(src, dst)
    print(f"Database successfully exported to: {dst}")


In [5]:
def merge_duplicate_columns(df):
    """
    Detects duplicate or suffixed columns (e.g. ID, ID:1, ID:2),
    merges them intelligently, and prints which columns were merged.
    """
    import re
    import pandas as pd

    # Identify base column names (e.g. 'ID' for 'ID:1')
    pattern = re.compile(r"^(.*?):\d+$")
    col_groups = {}

    for col in df.columns:
        match = pattern.match(col)
        base = match.group(1) if match else col
        col_groups.setdefault(base, []).append(col)

    # For each group of columns, merge if multiple versions exist
    for base, cols in col_groups.items():
        if len(cols) > 1:
            print(f"\nMerging columns for '{base}': {cols}")

            # Fill missing values from right to left (keep first non-null)
            df[base] = df[cols].bfill(axis=1).iloc[:, 0]

            # Check if all values are identical (for validation)
            all_equal = df[cols].nunique(axis=1).eq(1).all()
            if all_equal:
                print(f"All values identical across {cols}, keeping single column '{base}'.")
            else:
                print(f"Values differ across {cols}, filled missing values where needed.")

            # Drop duplicate columns except the merged one
            to_drop = [c for c in cols if c != base]
            df.drop(columns=to_drop, inplace=True)
            print(f"Dropped columns: {to_drop}")

    print("\nDuplicate column merging complete.")
    print(f"Final column count: {len(df.columns)}\n")
    return df



def join_tables(conn):
    """Create combined tables by joining person, vehicle, and accident tables for each dataset."""
    print("starting")
    cursor = conn.cursor()

    # Combine CRSS tables
    cursor.executescript("""
    DROP TABLE IF EXISTS crss_combined;
    CREATE TABLE crss_combined AS
    SELECT * FROM crss_person p
    LEFT JOIN crss_vehicle v
    ON p.CASENUM = v.CASENUM AND p.VEH_NO = v.VEH_NO
    LEFT JOIN crss_accident a
    ON p.CASENUM = a.CASENUM;
    """)

    print("continuing")
    # Combine FARS tables
    cursor.executescript("""
    DROP TABLE IF EXISTS fars_combined;
    CREATE TABLE fars_combined AS
    SELECT * FROM fars_person p
    LEFT JOIN fars_vehicle v
    ON p.ID = v.ID AND p.VEH_NO = v.VEH_NO
    LEFT JOIN fars_accident a
    ON p.ID = a.ID;
    """)

    conn.commit()

    print("almost there")

    # Load into pandas and merge duplicate columns
    crss_df = pd.read_sql("SELECT * FROM crss_combined", conn)
    fars_df = pd.read_sql("SELECT * FROM fars_combined", conn)

    crss_df = merge_duplicate_columns(crss_df)
    fars_df = merge_duplicate_columns(fars_df)

    # Replace old tables with cleaned versions
    crss_df.to_sql("crss_combined", conn, if_exists="replace", index=False)
    fars_df.to_sql("fars_combined", conn, if_exists="replace", index=False)

    print("Duplicate columns merged in CRSS and FARS combined tables.")


In [7]:
def main():
    crss_acc, crss_veh, crss_per, fars_acc, fars_veh, fars_per = load_cleaned_data()

    print("done 1")

    # Create SQLite database connection
    conn = sqlite3.connect("crss_fars_all.db")

    # Store and combine data
    write_to_database(conn, crss_acc, crss_veh, crss_per, fars_acc, fars_veh, fars_per)
    print("done 2")
    join_tables(conn)
    print("done 3")
    check_duplicates(conn)
    print("done 4")
    export_tables(conn, output_dir="output")
    print("done 5")

if __name__ == "__main__":
    main()


done 1
done 2
starting
continuing
almost there

Merging columns for 'ID': ['ID', 'ID:1', 'ID:2']
All values identical across ['ID', 'ID:1', 'ID:2'], keeping single column 'ID'.
Dropped columns: ['ID:1', 'ID:2']

Merging columns for 'CASENUM': ['CASENUM', 'CASENUM:1', 'CASENUM:2']
All values identical across ['CASENUM', 'CASENUM:1', 'CASENUM:2'], keeping single column 'CASENUM'.
Dropped columns: ['CASENUM:1', 'CASENUM:2']

Merging columns for 'VEH_NO': ['VEH_NO', 'VEH_NO:1']
All values identical across ['VEH_NO', 'VEH_NO:1'], keeping single column 'VEH_NO'.
Dropped columns: ['VEH_NO:1']

Merging columns for 'URBANICITY': ['URBANICITY', 'URBANICITY:1']
All values identical across ['URBANICITY', 'URBANICITY:1'], keeping single column 'URBANICITY'.
Dropped columns: ['URBANICITY:1']

Merging columns for 'REGION': ['REGION', 'REGION:1']
All values identical across ['REGION', 'REGION:1'], keeping single column 'REGION'.
Dropped columns: ['REGION:1']

Merging columns for 'YEAR': ['YEAR', 'YEAR

In [9]:
def check_missing_values_percent(df, label=None):
    """
    Report percent missing values for all columns in a DataFrame.
    Prints full column list (no truncation).
    """
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)

    # Compute percent missing
    missing_pct = df.isna().mean() * 100
    missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)

    print("\n" + "=" * 60)
    if label:
        print(f"[INFO] Percent missing summary for {label}:")
    else:
        print("[INFO] Percent missing summary:")
    print("=" * 60)

    if missing_pct.empty:
        print("No missing values found.")
    else:
        print(missing_pct.to_string(float_format="%.2f"))
    print("=" * 60 + "\n")

    return missing_pct


In [11]:
def check_original_missing(db_path="output/crss_fars_all.db"):
    conn = sqlite3.connect(db_path)

    fars = pd.read_sql("SELECT * FROM fars_combined", conn)
    crss = pd.read_sql("SELECT * FROM crss_combined", conn)

    print("\n--- Checking missing values in FARS ---")
    check_missing_values_percent(fars, label="FARS Original")

    print("\n--- Checking missing values in CRSS ---")
    check_missing_values_percent(crss, label="CRSS Original")

    conn.close()
    
check_original_missing()


--- Checking missing values in FARS ---

[INFO] Percent missing summary for FARS Original:
TRAV_SP        64.19
WORK_INJ       58.90
DRUGS          56.71
TRLR1GVWR      54.97
TRLR2GVWR      53.10
TRLR3GVWR      53.04
DRINKING       49.25
REST_USE       40.99
VPAVETYP       33.53
EJECTION       17.70
VTRAFCON       17.70
VPROFILE       17.70
AIR_BAG        17.69
VTCONT_F       17.41
SPEEDREL       14.00
VSPD_LIM       13.03
P_CRASH2       12.57
VALIGN         11.82
CARGO_BT       11.30
BODY_TYP       11.28
VSURCOND       11.28
P_CRASH1       11.13
V_CONFIG       10.94
VNUM_LAN       10.91
SEAT_POS       10.69
VTRAFWAY       10.41
TOW_VEH        10.15
M_HARM         10.00
WEATHER         5.20
WEATHERNAME     4.96
AGE             2.53
SEX             2.07
INJ_SEV         1.55
CITY            0.87
CITYNAME        0.87
HOUR            0.45
LGT_COND        0.38
LONGITUD        0.37
LATITUDE        0.37
TYP_INT         0.31
TYP_INTNAME     0.24
LGT_CONDNAME    0.19
RELJCT2         0.18
RUR_U

In [13]:
def print_all_columns(db_path="crss_fars_all.db"):
    """
    Print all tables and their columns (with data types) in a SQLite database.

    Parameters
    ----------
    db_path : str
        Path to the SQLite database file.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get all user-defined tables
    cursor.execute("""
        SELECT name FROM sqlite_master 
        WHERE type='table' AND name NOT LIKE 'sqlite_%';
    """)
    tables = [row[0] for row in cursor.fetchall()]

    if not tables:
        print("No user-defined tables found in the database.")
        conn.close()
        return

    print("\nDATABASE SCHEMA SUMMARY")
    print("=" * 70)
    for table in tables:
        cursor.execute(f"PRAGMA table_info({table});")
        cols = cursor.fetchall()
        print(f"\nTable: {table}")
        print("-" * 70)
        for col in cols:
            cid, name, col_type, notnull, dflt_value, pk = col
            pk_marker = " (PK)" if pk else ""
            print(f"{name} — {col_type}{pk_marker}")
    print("=" * 70)

    conn.close()

In [15]:
print_all_columns("output/crss_fars_all.db")



DATABASE SCHEMA SUMMARY

Table: crss_accident
----------------------------------------------------------------------
ID — TEXT
CASENUM — INTEGER
VE_TOTAL — INTEGER
MONTH — INTEGER
YEAR — INTEGER
DAY_WEEK — INTEGER
HOUR — REAL
RELJCT2 — REAL
TYP_INT — REAL
LGT_COND — REAL
WEATHER — REAL
URBANICITY — INTEGER
REGION — INTEGER

Table: crss_vehicle
----------------------------------------------------------------------
ID — TEXT
CASENUM — INTEGER
VEH_NO — INTEGER
VSURCOND — REAL
VTRAFCON — REAL
BODY_TYP — REAL
TOW_VEH — REAL
V_CONFIG — REAL
CARGO_BT — REAL
TRAV_SP — REAL
M_HARM — REAL
SPEEDREL — REAL
VTRAFWAY — REAL
VNUM_LAN — REAL
VSPD_LIM — REAL
VALIGN — REAL
VPROFILE — REAL
VTCONT_F — REAL
P_CRASH1 — REAL
P_CRASH2 — REAL
YEAR — INTEGER
TRLR1GVWR — REAL
TRLR2GVWR — REAL
TRLR3GVWR — REAL

Table: crss_person
----------------------------------------------------------------------
ID — TEXT
CASENUM — INTEGER
VEH_NO — INTEGER
PER_NO — INTEGER
STRATUM — INTEGER
AGE — REAL
SEX — REAL
PER_TYP — RE

In [17]:
def compare_fars_crss_columns(db_path="crss_fars_all.db"):
    """
    Compare column names between fars_combined and crss_combined tables.
    Prints columns unique to each and those shared by both.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Get column names from both tables
    cursor.execute("PRAGMA table_info(fars_combined);")
    fars_cols = [row[1] for row in cursor.fetchall()]

    cursor.execute("PRAGMA table_info(crss_combined);")
    crss_cols = [row[1] for row in cursor.fetchall()]

    # Convert to sets
    fars_set = set(fars_cols)
    crss_set = set(crss_cols)

    # Compare
    shared = fars_set & crss_set
    only_fars = fars_set - crss_set
    only_crss = crss_set - fars_set

    print("Columns present in BOTH FARS and CRSS:")
    print(sorted(shared))
    print("\nColumns only in FARS:")
    print(sorted(only_fars))
    print("\nColumns only in CRSS:")
    print(sorted(only_crss))
    print("\nSummary:")
    print(f"FARS total: {len(fars_cols)} | CRSS total: {len(crss_cols)} | Shared: {len(shared)}")

    conn.close()

In [19]:
compare_fars_crss_columns("output/crss_fars_all.db")


Columns present in BOTH FARS and CRSS:
['AGE', 'AIR_BAG', 'BODY_TYP', 'CARGO_BT', 'DAY_WEEK', 'DRINKING', 'DRUGS', 'EJECTION', 'HOUR', 'ID', 'INJ_SEV', 'LGT_COND', 'MONTH', 'M_HARM', 'PER_NO', 'PER_TYP', 'P_CRASH1', 'P_CRASH2', 'RELJCT2', 'REST_USE', 'SEAT_POS', 'SEX', 'SPEEDREL', 'TOW_VEH', 'TRAV_SP', 'TRLR1GVWR', 'TRLR2GVWR', 'TRLR3GVWR', 'TYP_INT', 'VALIGN', 'VEH_NO', 'VE_TOTAL', 'VNUM_LAN', 'VPROFILE', 'VSPD_LIM', 'VSURCOND', 'VTCONT_F', 'VTRAFCON', 'VTRAFWAY', 'V_CONFIG', 'WEATHER', 'YEAR']

Columns only in FARS:
['CITY', 'CITYNAME', 'COUNTY', 'COUNTYNAME', 'DAY', 'DAY_WEEKNAME', 'LATITUDE', 'LGT_CONDNAME', 'LONGITUD', 'MONTHNAME', 'RELJCT2NAME', 'RUR_URB', 'RUR_URBNAME', 'STATE', 'ST_CASE', 'TYP_INTNAME', 'VPAVETYP', 'WEATHERNAME', 'WORK_INJ']

Columns only in CRSS:
['CASENUM', 'REGION', 'STRATUM', 'URBANICITY', 'WEIGHT']

Summary:
FARS total: 61 | CRSS total: 47 | Shared: 42


In [21]:
def harmonize_and_merge_fars_crss(db_path="crss_fars_all.db", output_table="all_accidents_combined"):
    """
    Harmonize FARS and CRSS datasets:
    - Align column names
    - Recode URBANICITY ↔ RUR_URB
    - Create REGION variable for FARS
    - Drop *_NAME columns
    - Keep only shared + harmonized + selected unique columns
    - Add SOURCE column
    - Save merged table to SQLite
    - Print detailed summary
    """

    conn = sqlite3.connect(db_path)

    fars = pd.read_sql("SELECT * FROM fars_combined", conn)
    crss = pd.read_sql("SELECT * FROM crss_combined", conn)

    print(f"\nLoaded datasets:")
    print(f"   • FARS rows: {len(fars):,} | columns: {len(fars.columns)}")
    print(f"   • CRSS rows: {len(crss):,} | columns: {len(crss.columns)}")

    # --- Drop *_NAME columns from FARS ---
    fars_before = fars.shape[1]
    fars = fars.loc[:, ~fars.columns.str.contains("NAME$", case=False)]
    fars_after = fars.shape[1]
    print(f"Dropped {fars_before - fars_after} *_NAME columns from FARS (now {fars_after} columns)")

    # --- Harmonize urban/rural variable ---
    # CRSS: 1 Urban, 2 Rural → convert to FARS coding (1 Rural, 2 Urban, 6 Other)
    if "URBANICITY" in crss.columns:
        crss["RUR_URB"] = crss["URBANICITY"].map({1: 2, 2: 1})
        crss.drop(columns=["URBANICITY"], inplace=True)
        print("Harmonized URBANICITY → RUR_URB in CRSS")

    # --- Create REGION variable for FARS ---
    if "STATE" in fars.columns:
        def assign_region(state_code):
            if state_code in [23, 25, 33, 44, 50, 9, 34, 36, 42]:  # Northeast
                return 1
            elif state_code in [17, 18, 19, 20, 26, 27, 29, 31, 38, 39, 46, 55]:  # Midwest
                return 2
            elif state_code in [1, 5, 10, 11, 12, 13, 21, 22, 24, 28, 37, 40, 45, 47, 48, 51, 54, 72]:  # South
                return 3
            elif state_code in [2, 4, 6, 8, 15, 16, 30, 32, 35, 41, 49, 53, 56]:  # West
                return 4
            else:
                return None

        fars["REGION"] = fars["STATE"].apply(assign_region)
        print("Created REGION variable for FARS")

    # --- Add missing columns to ensure compatibility ---
    for col in ["LATITUDE", "LONGITUD"]:
        if col not in crss.columns:
            crss[col] = pd.NA
            print(f"Added missing column {col} to CRSS (filled with NA)")

    if "WEIGHT" not in fars.columns:
        fars["WEIGHT"] = pd.NA
        print("Added missing column WEIGHT to FARS (filled with NA)")

    # --- Add source column ---
    fars["SOURCE"] = "FARS"
    crss["SOURCE"] = "CRSS"

    # --- Determine shared/harmonized columns ---
    shared_cols = set(fars.columns) & set(crss.columns)
    harmonized_cols = sorted(shared_cols)
    print(f"\nFinal harmonized column count: {len(harmonized_cols)}")

    # --- Merge datasets ---
    combined = pd.concat([fars[harmonized_cols], crss[harmonized_cols]], axis=0, ignore_index=True)

    # --- Print summary ---
    print(f"\nMerge complete:")
    print(f"   • Combined total rows: {len(combined):,}")
    print(f"   • From FARS: {len(fars):,}")
    print(f"   • From CRSS: {len(crss):,}")
    print(f"   • Final columns: {len(combined.columns)}")

    # --- Save to SQLite ---
    combined.to_sql(output_table, conn, if_exists="replace", index=False)
    print(f"\nMaster table '{output_table}' saved to database '{db_path}'.")

    conn.close()
    return combined


In [23]:
harmonize_and_merge_fars_crss("output/crss_fars_all.db")


Loaded datasets:
   • FARS rows: 711,994 | columns: 61
   • CRSS rows: 1,032,571 | columns: 47
Dropped 9 *_NAME columns from FARS (now 52 columns)
Harmonized URBANICITY → RUR_URB in CRSS
Created REGION variable for FARS
Added missing column LATITUDE to CRSS (filled with NA)
Added missing column LONGITUD to CRSS (filled with NA)
Added missing column WEIGHT to FARS (filled with NA)

Final harmonized column count: 48


C:\Users\brigi\AppData\Local\Temp\ipykernel_26152\1102588238.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([fars[harmonized_cols], crss[harmonized_cols]], axis=0, ignore_index=True)



Merge complete:
   • Combined total rows: 1,744,565
   • From FARS: 711,994
   • From CRSS: 1,032,571
   • Final columns: 48

Master table 'all_accidents_combined' saved to database 'output/crss_fars_all.db'.


KeyboardInterrupt: 

In [11]:
combined = harmonize_and_merge_fars_crss("output/crss_fars_all.db")

check_missing_values(combined, label="FARS + CRSS Combined Table")


Loaded datasets:
   • FARS rows: 711,994 | columns: 61
   • CRSS rows: 1,032,571 | columns: 47
Dropped 9 *_NAME columns from FARS (now 52 columns)
Harmonized URBANICITY → RUR_URB in CRSS
Created REGION variable for FARS
Added missing column LATITUDE to CRSS (filled with NA)
Added missing column LONGITUD to CRSS (filled with NA)
Added missing column WEIGHT to FARS (filled with NA)

Final harmonized column count: 48


C:\Users\brigi\AppData\Local\Temp\ipykernel_36632\1102588238.py:73: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat([fars[harmonized_cols], crss[harmonized_cols]], axis=0, ignore_index=True)



Merge complete:
   • Combined total rows: 1,744,565
   • From FARS: 711,994
   • From CRSS: 1,032,571
   • Final columns: 48

Master table 'all_accidents_combined' saved to database 'output/crss_fars_all.db'.

[INFO] Missing value summary for FARS + CRSS Combined Table:
AGE            83781
AIR_BAG       263923
BODY_TYP      767490
CARGO_BT      778456
DRINKING      770574
DRUGS         920254
EJECTION      244537
HOUR            6687
INJ_SEV        49609
LATITUDE     1035229
LGT_COND        7812
LONGITUD     1035229
M_HARM        754608
PER_TYP         1098
P_CRASH1      769559
P_CRASH2      771915
RELJCT2        47132
REST_USE      474961
RUR_URB          939
SEAT_POS      128193
SEX            59222
SPEEDREL      774021
TOW_VEH       755559
TRAV_SP      1269401
TRLR1GVWR    1259711
TRLR2GVWR    1247058
TRLR3GVWR    1246767
TYP_INT        86404
VALIGN        807860
VNUM_LAN     1018045
VPROFILE      893859
VSPD_LIM      891420
VSURCOND      803441
VTCONT_F      852339
VTRAFCON      

AGE            83781
AIR_BAG       263923
BODY_TYP      767490
CARGO_BT      778456
DRINKING      770574
DRUGS         920254
EJECTION      244537
HOUR            6687
INJ_SEV        49609
LATITUDE     1035229
LGT_COND        7812
LONGITUD     1035229
M_HARM        754608
PER_TYP         1098
P_CRASH1      769559
P_CRASH2      771915
RELJCT2        47132
REST_USE      474961
RUR_URB          939
SEAT_POS      128193
SEX            59222
SPEEDREL      774021
TOW_VEH       755559
TRAV_SP      1269401
TRLR1GVWR    1259711
TRLR2GVWR    1247058
TRLR3GVWR    1246767
TYP_INT        86404
VALIGN        807860
VNUM_LAN     1018045
VPROFILE      893859
VSPD_LIM      891420
VSURCOND      803441
VTCONT_F      852339
VTRAFCON      855270
VTRAFWAY      916431
V_CONFIG      770279
WEATHER        77862
WEIGHT        711994
dtype: int64